In [0]:
%run ../01_Bronze/00_setup

In [0]:
# process new records into the Silver layer shelves table
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, current_date

def process_scd2_shelves(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.shelves"
    
    batch_df = microBatchDF.orderBy(F.col("updated_timestamp").desc()) \
                           .dropDuplicates(["shelf_id"])
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = batch_df.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table) \
        .filter("is_active = true") \
        .select("shelf_id")
    
    updates_df = batch_df.join(existing_ids, "shelf_id", "inner") \
        .withColumn("merge_key", F.col("shelf_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    #combined_df.createOrReplaceTempView("source_shelves_view")
    view_name = f"source_shelves_{batchId}"
    combined_df.createOrReplaceTempView(view_name)
    try:
        microBatchDF.sparkSession.sql(f"""
            MERGE INTO {target_table} AS target
            USING {view_name} AS source
            ON target.shelf_id = source.merge_key AND target.is_active = true
            
            -- If we find the ID and status changed, expire the old one
            WHEN MATCHED AND (target.status <> source.status OR NOT(target.robot_id <=> source.robot_id)) THEN
            UPDATE SET target.is_active = false, target.end_date = current_timestamp(), target.updated_timestamp = current_timestamp()
            
            -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
            WHEN NOT MATCHED THEN
            INSERT (shelf_sk, shelf_id, robot_id, max_weight_capacity, status, is_active, start_date, end_date, updated_timestamp)
            VALUES (md5(concat(source.shelf_id, cast(source.updated_timestamp as STRING))), 
                    source.shelf_id, source.robot_id, source.max_weight_capacity, source.status,  
                    true, source.updated_timestamp, NULL, current_timestamp())
        """)
    except Exception as e:
        print(f"Error in batch {batchId}: {e}")
        raise e

    microBatchDF.sparkSession.catalog.dropTempView(view_name)
    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.shelves_events")
    .filter("status IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd2_shelves)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_shelves_scd2")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())

In [0]:
%sql
select * from workspace.amazon_fullfilment_silver.shelves